# 02 — Logistic Regression Baseline

Establish an interpretable baseline on the training and validation partitions only. The held-out test labels remain unused until the final model and threshold are frozen.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_dataset, split_dataset
from src.evaluation import best_f1_threshold, classification_metrics
from src.models import build_logistic_regression
from src.preprocessing import build_linear_preprocessor

METRICS = ROOT / 'outputs' / 'metrics'
METRICS.mkdir(parents=True, exist_ok=True)

In [2]:
transactions = load_dataset(ROOT / 'data' / 'creditcard.csv')
X_train, X_validation, X_test, y_train, y_validation, y_test = split_dataset(transactions)

preprocessor = build_linear_preprocessor(X_train.columns.tolist())
X_train_scaled = preprocessor.fit_transform(X_train)
X_validation_scaled = preprocessor.transform(X_validation)

baseline = build_logistic_regression()
baseline.fit(X_train_scaled, y_train)
validation_scores = baseline.predict_proba(X_validation_scaled)[:, 1]

In [3]:
default_metrics = classification_metrics(y_validation, validation_scores, threshold=0.5)
validation_threshold = best_f1_threshold(y_validation, validation_scores)
tuned_metrics = classification_metrics(y_validation, validation_scores, threshold=validation_threshold)

baseline_results = {
    'default_threshold': 0.5,
    'default_metrics': default_metrics,
    'validation_selected_threshold': validation_threshold,
    'validation_metrics_at_selected_threshold': tuned_metrics,
}
baseline_results

{'default_threshold': 0.5,
 'default_metrics': {'roc_auc': 0.9806087815074327,
  'pr_auc': 0.6858393374763166,
  'precision': 0.06413124533929904,
  'recall': 0.8775510204081632,
  'f1': 0.11952744961779013,
  'tn': 55608,
  'fp': 1255,
  'fn': 12,
  'tp': 86},
 'validation_selected_threshold': 0.9999999980334442,
 'validation_metrics_at_selected_threshold': {'roc_auc': 0.9806087815074327,
  'pr_auc': 0.6858393374763166,
  'precision': 0.8604651162790697,
  'recall': 0.7551020408163265,
  'f1': 0.8043478260869565,
  'tn': 56851,
  'fp': 12,
  'fn': 24,
  'tp': 74}}

In [4]:
with (METRICS / 'logistic_baseline_validation.json').open('w') as file:
    json.dump(baseline_results, file, indent=2)

print(f"Validation PR-AUC: {default_metrics['pr_auc']:.4f}")
print(f"Best validation F1 threshold: {validation_threshold:.4f}")
print('Test labels were not evaluated in this notebook.')

Validation PR-AUC: 0.6858
Best validation F1 threshold: 1.0000
Test labels were not evaluated in this notebook.


## Interpretation

The logistic model provides the benchmark reference for more expressive classifiers. Class weighting addresses the rare positive class during fitting, while threshold selection is performed on validation data rather than assuming that `0.5` is operationally appropriate. No test-set performance is reported at this stage.